# Vincent Straussberger

## Inference Model

This notebook uses `Qwen/Qwen3-0.6B` for direct question answering on the Austrian tax benchmark.

Key facts:
- model: `Qwen/Qwen3-0.6B`
- input: `dataset_clean.csv`
- output: one submission CSV with `id,answer`


## How the code works

- loads and validates the benchmark dataset
- formats each question for the model
- generates one answer per row
- saves the final CSV in this folder


In [ ]:
print('Skip this cell if your packages are already installed.')
print('Minimal install for this notebook: python -m pip install pandas numpy torch transformers sentence-transformers beautifulsoup4 pypdf requests tqdm')

In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    'pandas': 'pandas',
    'requests': 'requests',
    'torch': 'torch',
    'tqdm': 'tqdm',
    'transformers': 'transformers',
}

missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('All required packages are already installed.')


Installing missing packages: requests


In [2]:
import json
import os
import statistics
import time
import warnings
from pathlib import Path

import pandas as pd
import requests
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

warnings.filterwarnings('ignore')
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


d:\DS Applications\.venv-gpu312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.11.0+cu128
CUDA available: True


In [3]:
def ensure_directory(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

CURRENT_DIR = Path.cwd().resolve()
REMOTE_DATASET_URL = 'https://raw.githubusercontent.com/svakulenk0/wu-llms-ss26/main/dataset_clean.csv'


def resolve_code_dir(current_dir: Path) -> Path:
    if current_dir.name == 'code':
        return current_dir
    candidate = current_dir / 'code'
    if candidate.exists():
        return candidate
    return current_dir


def resolve_submission_root(code_dir: Path) -> Path:
    if code_dir.name == 'code':
        return code_dir.parent
    if (code_dir / 'code').exists():
        return code_dir
    return code_dir.parent


def ensure_local_dataset(submission_root: Path, remote_url: str, filename: str = 'dataset_clean.csv') -> Path:
    local_path = submission_root / filename
    if local_path.exists():
        return local_path
    try:
        response = requests.get(remote_url, timeout=30)
        response.raise_for_status()
        local_path.write_text(response.text, encoding='utf-8')
        print(f'Downloaded benchmark dataset to {local_path}')
        return local_path
    except Exception as exc:
        raise FileNotFoundError(
            f'Could not find {filename} in the submission root and the download also failed. Original error: {exc}'
        ) from exc


NOTEBOOK_DIR = resolve_code_dir(CURRENT_DIR)
SUBMISSION_ROOT = resolve_submission_root(NOTEBOOK_DIR)
RESULT_DIR = ensure_directory(SUBMISSION_ROOT / 'results')
LOCAL_ASSET_DIR = NOTEBOOK_DIR / '_inference_assets'
MODEL_CACHE_DIR = LOCAL_ASSET_DIR / 'model_cache'
DATASET_PATH = ensure_local_dataset(NOTEBOOK_DIR, REMOTE_DATASET_URL)
MODEL_NAME = 'Qwen/Qwen3-0.6B'
# Slightly older but proven fallback:
# MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
RESULT_PATH = RESULT_DIR / 'inference_qwen3_0_6b.csv'
FAILURE_LOG_PATH = RESULT_DIR / 'inference_qwen3_0_6b_failures.json'

ROW_LIMIT = None
CHECK_REMOTE_DATASET = False
FAIL_ON_REMOTE_MISMATCH = False
SAVE_EVERY_N_ROWS = 5
MAX_INPUT_TOKENS = 420
MAX_NEW_TOKENS = 160
TEMPERATURE = 0.7
TOP_P = 0.8
DO_SAMPLE = True
REPETITION_PENALTY = 1.05
MAX_RETRIES = 2
MIN_ANSWER_LENGTH = 20
SEED = 42

set_seed(SEED)
ensure_directory(LOCAL_ASSET_DIR)
ensure_directory(MODEL_CACHE_DIR)

print('Current directory:', CURRENT_DIR)
print('Notebook directory:', NOTEBOOK_DIR)
print('Submission root:', SUBMISSION_ROOT)
print('Dataset path:', DATASET_PATH)
print('Result path:', RESULT_PATH)
print('Model:', MODEL_NAME)


Notebook directory: D:\DS Applications\Vincent
Dataset path: D:\DS Applications\Vincent\dataset_clean.csv
Result path: D:\DS Applications\Vincent\inference_qwen3_0_6b.csv
Model: Qwen/Qwen3-0.6B


In [4]:
REQUIRED_COLUMNS = {'id', 'prompt'}
SYSTEM_PROMPT = (
    'Du bist ein hilfreicher Assistent fuer oesterreichisches Steuerrecht. '
    'Antworte auf Deutsch, kurz und praezise. '
    'Erfinde keine Fakten. Wenn du unsicher bist, sage das knapp.'
)


def normalize_answer(value) -> str:
    return ' '.join(str(value or '').strip().split())


def retry_call(fn, attempts: int = 2):
    last_error = None
    for _ in range(attempts):
        try:
            return fn()
        except Exception as exc:
            last_error = exc
    raise last_error


def read_remote_row_count() -> int | None:
    try:
        response = requests.get(REMOTE_DATASET_URL, timeout=30)
        response.raise_for_status()
        remote_frame = pd.read_csv(pd.io.common.StringIO(response.text), encoding='utf-8-sig')
        return len(remote_frame)
    except Exception as exc:
        print(f'Remote dataset check skipped: {exc}')
        return None


def load_benchmark_rows(dataset_path: Path, row_limit: int | None = None) -> list[dict]:
    if not dataset_path.exists():
        raise FileNotFoundError(f'Dataset not found: {dataset_path}')

    frame = pd.read_csv(dataset_path, encoding='utf-8-sig', dtype={'id': 'string', 'prompt': 'string'}, low_memory=False)
    missing_columns = REQUIRED_COLUMNS - set(frame.columns)
    if missing_columns:
        raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

    if frame['id'].duplicated().any():
        duplicate_ids = frame.loc[frame['id'].duplicated(), 'id'].astype(str).tolist()
        raise ValueError(f'Duplicate ids found in dataset: {duplicate_ids[:10]}')

    rows = []
    source_frame = frame.head(row_limit).copy() if row_limit is not None else frame.copy()
    for row in source_frame.itertuples(index=False):
        question = normalize_answer(row.prompt)
        if question:
            rows.append({'id': str(row.id), 'prompt': question})
    if not rows:
        raise ValueError('No usable benchmark rows were found.')
    return rows


def load_existing_submission(result_path: Path) -> dict[str, str]:
    if not result_path.exists():
        return {}
    frame = pd.read_csv(result_path, encoding='utf-8-sig', dtype={'id': 'string', 'answer': 'string'})
    if list(frame.columns) != ['id', 'answer']:
        raise ValueError('Existing submission file must have exactly the columns id,answer')
    if frame['id'].duplicated().any():
        raise ValueError('Existing submission contains duplicate ids.')
    return {str(row.id): normalize_answer(row.answer) for row in frame.itertuples(index=False) if normalize_answer(row.answer)}


def write_submission(ordered_rows: list[tuple[str, str]], result_path: Path):
    result_frame = pd.DataFrame(ordered_rows, columns=['id', 'answer'])
    result_frame.to_csv(result_path, index=False, encoding='utf-8-sig')


def validate_submission(benchmark_rows: list[dict], answers: dict[str, str], min_answer_length: int = 1) -> dict:
    expected_ids = [row['id'] for row in benchmark_rows]
    missing_ids = [row_id for row_id in expected_ids if row_id not in answers]
    extra_ids = [row_id for row_id in answers if row_id not in set(expected_ids)]
    short_answer_ids = [row_id for row_id in expected_ids if row_id in answers and len(normalize_answer(answers[row_id])) < min_answer_length]
    return {
        'row_count': len(answers),
        'missing_ids': missing_ids,
        'extra_ids': extra_ids,
        'short_answer_ids': short_answer_ids,
    }


def pick_device():
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def render_model_prompt(tokenizer, question: str) -> str:
    cleaned_question = normalize_answer(question)
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': cleaned_question}]
    if getattr(tokenizer, 'chat_template', None):
        if MODEL_NAME.startswith('Qwen/Qwen3'):
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f'Instruction: {SYSTEM_PROMPT}\nQuestion: {cleaned_question}\nAnswer:'


def load_model_and_tokenizer(model_name: str):
    try:
        device = pick_device()
        if device.type == 'cpu':
            torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))

        tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=str(MODEL_CACHE_DIR), use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model_kwargs = {
            'cache_dir': str(MODEL_CACHE_DIR),
            'low_cpu_mem_usage': True,
        }
        if device.type in {'cuda', 'mps'}:
            model_kwargs['torch_dtype'] = 'auto'

        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        model.to(device)
        model.eval()
        return model, tokenizer, device
    except Exception as exc:
        raise RuntimeError(f'Could not load the model/tokenizer for {model_name}: {exc}') from exc


def generate_answer(model, tokenizer, device, question: str) -> str:
    full_prompt = render_model_prompt(tokenizer, question)
    token_batch = tokenizer(
        full_prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    token_batch = {name: value.to(device) for name, value in token_batch.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **token_batch,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=DO_SAMPLE,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_length = token_batch['attention_mask'].sum(dim=1).item()
    answer_ids = generated_ids[0][int(prompt_length):]
    answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
    return normalize_answer(answer_text)


In [5]:
benchmark_rows = load_benchmark_rows(DATASET_PATH, row_limit=ROW_LIMIT)
existing_answers = load_existing_submission(RESULT_PATH)
pending_rows = [row for row in benchmark_rows if row['id'] not in existing_answers]

if CHECK_REMOTE_DATASET:
    remote_row_count = read_remote_row_count()
    if remote_row_count is not None and remote_row_count != len(pd.read_csv(DATASET_PATH, encoding='utf-8-sig')):
        message = f'Local dataset row count does not match the remote dataset: local={len(pd.read_csv(DATASET_PATH, encoding="utf-8-sig"))}, remote={remote_row_count}'
        if FAIL_ON_REMOTE_MISMATCH:
            raise ValueError(message)
        print('Warning:', message)

print('Rows loaded:', len(benchmark_rows))
print('Already answered:', len(existing_answers))
print('Pending rows:', len(pending_rows))


Rows loaded: 643
Already answered: 0
Pending rows: 643


In [6]:
model, tokenizer, device = load_model_and_tokenizer(MODEL_NAME)
print('Model and tokenizer are ready on', device)

timings = []
failures = {}

for row_index, row in enumerate(tqdm(pending_rows, desc='Inference'), start=1):
    started = time.perf_counter()
    try:
        answer = retry_call(lambda row=row: generate_answer(model, tokenizer, device, row['prompt']), attempts=MAX_RETRIES)
        if answer:
            existing_answers[row['id']] = answer
        else:
            failures[row['id']] = 'The model returned an empty answer after cleaning.'
    except Exception as exc:
        failures[row['id']] = str(exc)

    timings.append(time.perf_counter() - started)

    if row_index % SAVE_EVERY_N_ROWS == 0 or row_index == len(pending_rows):
        ordered_rows = [(item['id'], existing_answers[item['id']]) for item in benchmark_rows if item['id'] in existing_answers]
        write_submission(ordered_rows, RESULT_PATH)
        FAILURE_LOG_PATH.write_text(json.dumps(failures, indent=2, ensure_ascii=False), encoding='utf-8')

print('Inference loop finished.')


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 8051.73it/s]


Model and tokenizer are ready on cuda


Inference: 100%|██████████| 643/643 [27:26<00:00,  2.56s/it]

Inference loop finished.


In [7]:
validation = validate_submission(benchmark_rows, existing_answers, min_answer_length=MIN_ANSWER_LENGTH)
summary = {
    'rows_written': validation['row_count'],
    'failed_rows': len(failures),
    'average_row_seconds': round(statistics.mean(timings), 3) if timings else 0.0,
    'short_answer_count': len(validation['short_answer_ids']),
    'missing_ids': validation['missing_ids'][:10],
    'result_path': str(RESULT_PATH),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('\nShort answers worth reviewing:', validation['short_answer_ids'][:10])
pd.read_csv(RESULT_PATH, encoding='utf-8-sig').head(5)


{
  "rows_written": 643,
  "failed_rows": 0,
  "average_row_seconds": 2.559,
  "short_answer_count": 2,
  "missing_ids": [],
  "result_path": "D:\\DS Applications\\Vincent\\inference_qwen3_0_6b.csv"
}

Short answers worth reviewing: ['VAT-INTL-049', 'EStG-23-058']


,id,answer
0,CORP-TAX-001,Die steuerliche Bemessungsgrundlage für die Kö...
1,CORP-TAX-002,Wenn eine Körperschaft ein zinsloses Darlehen ...
2,CORP-TAX-003,"Die Körperschaften, die sämtliche Einkünfte au..."
3,CORP-TAX-004,(a) Bei der Tochtergesellschaft wird die Divid...
4,CORP-TAX-005,Die steuerliche Behandlung einer offenen Aussc...


## Ergebnis

Die CSV wird im Modellordner als `inference_qwen3_0_6b.csv` gespeichert.
